# Football Analytics Package — Demo Notebook

This notebook demonstrates the full capabilities of the `football-analytics` package.

We use two data sources:
- **StatsBomb open data** — player-level event data, xG, passes, defensive actions
- **football-data.org** — standings, fixtures, match results

### Sections
1. Setup and imports
2. Exploring available StatsBomb data
3. Match level analysis
4. Player comparison — key players across La Liga seasons
5. Season level analysis and comparison

### 1. Setup and imports 

In [27]:
# standard library
import warnings

# data
import pandas as pd

warnings.filterwarnings("ignore")

# our package — data layer
from football_analytics.data import FootballDataClient, StatsBombClient  # noqa: E402, I001

# our package — analytics layer
from football_analytics.analytics import (  # noqa: E402
    # form
    get_recent_form,
    get_points_per_game,
    get_home_away_split,
    # xG
    get_match_xg_summary,
    get_player_xg_ranking,
    get_xg_overperformance,
    # standings
    get_clean_standings,
    get_expected_vs_actual,
    # player
    get_top_performers,
    add_per_90_columns,
    per_90,
)

print("imports OK")

imports OK


In [28]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
# Initialise clients 

# StatsBomb requires no API key 
sb_client = StatsBombClient()

# FootballDataClient reads the key from .env aautomatically 
fd_client = FootballDataClient()

print("Clients initialised OK")

Clients initialised OK


### 2. Exploring available StatsBomb 

First we can look at what data from which competitions are contained in the open package of StatsBomb.

In [30]:
competitions = sb_client.get_competitions()
competitions

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882
...,...,...,...,...,...,...,...,...,...,...,...,...
75,35,75,Europe,UEFA Europa League,male,False,False,1988/1989,2026-04-11T12:48:10.012987,2021-06-13T16:17:31.694,NaN,2026-04-11T12:48:10.012987
76,53,315,Europe,UEFA Women's Euro,female,False,True,2025,2026-04-27T22:02:42.690507,2026-04-27T22:03:28.087062,2026-04-27T22:03:28.087062,2026-04-27T22:02:42.690507
77,53,106,Europe,UEFA Women's Euro,female,False,True,2022,2026-05-05T03:03:04.199896,2026-05-05T03:05:32.480837,2026-05-05T03:05:32.480837,2026-05-05T03:03:04.199896
78,72,107,International,Women's World Cup,female,False,True,2023,2026-05-03T13:51:31.021141,2026-05-03T13:55:52.303219,2026-05-03T13:55:52.303219,2026-05-03T13:51:31.021141


In [31]:
# Let's see all the available competitions  
competitions.competition_name.unique()

<StringArray>
[          '1. Bundesliga',  'African Cup of Nations',
        'Champions League',            'Copa America',
            'Copa del Rey', 'FA Women's Super League',
      'FIFA U20 World Cup',          'FIFA World Cup',
       'Frauen Bundesliga',     'Indian Super league',
                 'La Liga',                  'Liga F',
        'Liga Profesional',                 'Ligue 1',
     'Major League Soccer',   'North American League',
                    'NWSL',          'Premier League',
                 'Serie A',           'Serie A Women',
               'UEFA Euro',      'UEFA Europa League',
       'UEFA Women's Euro',       'Women's World Cup']
Length: 24, dtype: str

In [32]:
# As a domain knowledge, we're aware that open StatsBomb data contains
# a lot of La Liga data.  Below is shown all the available seasons for LaLiga
 
bundesliga = competitions[
    competitions["competition_name"].str.contains("liga", case=False)
]
print(bundesliga[["competition_id", "season_id", "competition_name", "season_name"]])


    competition_id  season_id   competition_name season_name
0                9        281      1. Bundesliga   2023/2024
1                9         27      1. Bundesliga   2015/2016
38             135        281  Frauen Bundesliga   2023/2024
40              11         90            La Liga   2020/2021
41              11         42            La Liga   2019/2020
42              11          4            La Liga   2018/2019
43              11          1            La Liga   2017/2018
44              11          2            La Liga   2016/2017
45              11         27            La Liga   2015/2016
46              11         26            La Liga   2014/2015
47              11         25            La Liga   2013/2014
48              11         24            La Liga   2012/2013
49              11         23            La Liga   2011/2012
50              11         22            La Liga   2010/2011
51              11         21            La Liga   2009/2010
52              11      

### 3. Match Level Analysis 

We start our data exploration from the smallest possible unit of comparison for our package, and that is a single match. Here we demonstrate which match-level analyses can be done with the package. 

We use a match from La Liga season 2015/16 (competition_id=11, season_id=27).

In [33]:
# retrieving all the matches in the given season 
COMPETITION_ID = 11 
SEASON_ID = 27 # 2015/16 

laliga_matches_15 = sb_client.get_matches(competition_id=COMPETITION_ID,
                                          season_id=SEASON_ID)

print ( f"Total matches in 2015/16: {len(laliga_matches_15)}")
laliga_matches_15[["match_id", "home_team", "away_team", "home_score", "away_score"]].head(10)

Total matches in 2015/16: 380


,match_id,home_team,away_team,home_score,away_score
0,3825739,Real Madrid,Sporting Gijón,5,1
1,3825848,Levante UD,Eibar,2,2
2,3825895,Las Palmas,Sevilla,2,0
3,3825894,RC Deportivo La Coruña,Getafe,0,2
4,3825855,Málaga,Levante UD,3,1
5,3825908,Espanyol,Eibar,4,2
6,3825883,Málaga,Las Palmas,4,1
7,3825900,Sporting Gijón,Villarreal,2,0
8,3825902,Rayo Vallecano,Levante UD,3,1
9,3825876,Real Betis,Getafe,2,1


In [34]:
# picking one match to explore from Barcelona matches 

barca_matches = sb_client.get_matches(competition_id=COMPETITION_ID,
                                      season_id=SEASON_ID,
                                      team="Barcelona")

print(f"Barcelona matches available: expected -> 38 retrieved -> {len(barca_matches)}")
barca_matches[["match_id", "home_team", "away_team", "home_score", "away_score"]].head(10)

Barcelona matches available: expected -> 38 retrieved -> 38


,match_id,home_team,away_team,home_score,away_score
358,266236,Athletic Club,Barcelona,0,1
357,267422,Barcelona,Málaga,1,0
371,266166,Atlético Madrid,Barcelona,1,2
377,266490,Barcelona,Levante UD,4,1
375,266467,Celta Vigo,Barcelona,4,1
356,267611,Barcelona,Las Palmas,2,1
306,3825617,Sevilla,Barcelona,2,1
300,3825627,Barcelona,Rayo Vallecano,5,2
186,3825637,Barcelona,Eibar,3,1
288,3825645,Getafe,Barcelona,0,2


#### 3.1 xG Summary for a Single Match 

`get_shots()` returns the raw shot data for a match.

`get_match_xg_summary()` takes raw shot data and returns a team-level summary of total shots, shots on target, xG and goals. 

xg_difference tells us how much a team over or underperformed their chances. 

In [35]:
# use the Barca-Rayo Vallecano match because it has a very high score and probably a lot
# of data to showcase 
MATCH_ID = barca_matches.iloc[3]["match_id"]
home = barca_matches.iloc[3]["home_team"]
away = barca_matches.iloc[3]["away_team"]
home_score = barca_matches.iloc[3]["home_score"]
away_score = barca_matches.iloc[3]["away_score"]

print(f"Match: {home} {home_score} - {away_score} {away}\n")

shots = sb_client.get_shots(match_id=MATCH_ID)
xg_summary = get_match_xg_summary(shots)
xg_summary

Match: Barcelona 4 - 1 Levante UD



,team,total_shots,shots_on_target,total_xg,goals,xg_difference
0,Barcelona,20,7,3.530,4,0.470
1,Levante UD,11,4,0.721,1,0.279


#### 3.2 Player Shooting Stats for a Single Match 

`get_player_shooting_match()` retruns player level xG data for a single match with data on shots taken, shots on target, goals scored and xG generated

In [36]:
shooting_stats = sb_client.get_player_shooting_match(match_id=MATCH_ID)
shooting_stats.sort_values('total_xg', ascending=False)


,player,team,position,shots,shots_on_target,goals,total_xg,xg_per_shot
5,Lionel Andrés Messi Cuccittini,Barcelona,Center Attacking Midfield,10,4,2,2.150,0.215
9,Neymar da Silva Santos Junior,Barcelona,Left Wing,4,1,1,0.561,0.140
6,Marc Bartra Aregall,Barcelona,Right Center Back,2,1,1,0.403,0.202
7,Munir El Haddadi Mohamed,Barcelona,Right Wing,2,1,0,0.280,0.140
13,Víctor Casadesús Castaño,Levante UD,Right Center Forward,1,1,1,0.180,0.180
3,Jefferson Andrés Lerma Solís,Levante UD,Left Center Midfield,2,0,0,0.162,0.081
12,Víctor Camarasa Ferrando,Levante UD,Right Center Midfield,2,0,0,0.161,0.081
11,Sandro Ramírez Castillo,Barcelona,Center Forward,1,0,0,0.116,0.116
8,Nabil Ghilas,Levante UD,Right Center Forward,1,0,0,0.098,0.098
0,Antonio García Aranda,Levante UD,Left Wing Back,1,0,0,0.045,0.045


#### 3.3 Player Goals & Assists for a Single Match 

`get_player_goals_assists_match()` returns the goals and assists for a single match

In [37]:
goals_assists = sb_client.get_player_goals_assists_match(match_id=MATCH_ID)
goals_assists.sort_values('goals',ascending=False)

,player,team,position,goals,assists
1,Lionel Andrés Messi Cuccittini,Barcelona,Center Attacking Midfield,2,1
3,Neymar da Silva Santos Junior,Barcelona,Left Wing,1,0
2,Marc Bartra Aregall,Barcelona,Right Center Back,1,1
4,Víctor Casadesús Castaño,Levante UD,Right Center Forward,1,0
0,Antonio García Aranda,Levante UD,Left Wing Back,0,1


#### 3.4 Player xG Ranking for a Single Match 

`get_player_xg_ranking()` ranks every player by their total xG generated
in the match. Useful for identifying who created the best chances regardless
of whether they scored.

In [38]:
xg_ranking = get_player_xg_ranking(shots)
xg_ranking

,player,team,shots,goals,total_xg,xg_per_shot
0,Lionel Andrés Messi Cuccittini,Barcelona,10,2,2.150,0.215
1,Neymar da Silva Santos Junior,Barcelona,4,1,0.561,0.140
2,Marc Bartra Aregall,Barcelona,2,1,0.403,0.202
3,Munir El Haddadi Mohamed,Barcelona,2,0,0.280,0.140
4,Víctor Casadesús Castaño,Levante UD,1,1,0.180,0.180
5,Jefferson Andrés Lerma Solís,Levante UD,2,0,0.162,0.081
6,Víctor Camarasa Ferrando,Levante UD,2,0,0.161,0.081
7,Sandro Ramírez Castillo,Barcelona,1,0,0.116,0.116
8,Nabil Ghilas,Levante UD,1,0,0.098,0.098
9,Antonio García Aranda,Levante UD,1,0,0.045,0.045


#### 3.5 Player Passing Stats for a Single Match 

`get_player_passing_match()` returns total passes, completed passes, completion rates and progressive passes per player for a single match. 

In [39]:
passing = sb_client.get_player_passing_match(match_id=MATCH_ID)
passing.sort_values('passes', ascending=False).head(10)

,player,team,position,passes,passes_completed,progressive_passes,completion_rate
2,Daniel Alves da Silva,Barcelona,Right Back,121,105,0,86.8
5,Ivan Rakitić,Barcelona,Right Defensive Midfield,89,81,0,91.0
12,Marc Bartra Aregall,Barcelona,Right Center Back,75,66,0,88.0
0,Adriano Correia Claro,Barcelona,Left Back,74,66,0,89.2
7,Javier Alejandro Mascherano,Barcelona,Left Center Back,73,72,0,98.6
16,Neymar da Silva Santos Junior,Barcelona,Left Wing,63,42,0,66.7
11,Lionel Andrés Messi Cuccittini,Barcelona,Center Attacking Midfield,61,49,0,80.3
14,Munir El Haddadi Mohamed,Barcelona,Right Wing,52,45,0,86.5
9,José Antonio García Rabasco,Levante UD,Center Defensive Midfield,51,38,0,74.5
21,Sergio Busquets i Burgos,Barcelona,Left Defensive Midfield,41,40,0,97.6


#### 3.6 Player Defensive Stats for a Single Match 

`get_player_defensive_match()` returns tackles, interceptions and clearances per player. This method allows us to see the defensive contributions from each player

In [40]:
defensive = sb_client.get_player_defensive_match(match_id=MATCH_ID)

defensive.sort_values('interceptions',ascending=False).head(10)

,player,team,position,tackles,interceptions,clearances
9,Juan Francisco García García,Levante UD,Left Center Back,0,4,5
3,Deyverson Brum Silva Acosta,Levante UD,Left Center Forward,0,3,0
15,Víctor Camarasa Ferrando,Levante UD,Right Center Midfield,0,3,0
7,Jefferson Andrés Lerma Solís,Levante UD,Left Center Midfield,0,2,0
10,Marc Bartra Aregall,Barcelona,Right Center Back,0,1,4
2,Daniel Alves da Silva,Barcelona,Right Back,0,1,1
11,Munir El Haddadi Mohamed,Barcelona,Right Wing,0,1,0
6,Iván López Mendoza,Levante UD,Right Wing Back,0,1,4
5,Ivan Rakitić,Barcelona,Right Defensive Midfield,0,1,0
8,José Antonio García Rabasco,Levante UD,Center Defensive Midfield,0,1,0


### 4. Player Comparison Across La Liga Seasons 

With this package, we can go even deeper into player analysis by being able to compare statistics of multiple players across seasons.  Here we compare three key players across three positions: 
- **Forwards:** Messi, Suárez, Ronaldo
- **Midfielders:** Modrić, Busquets, David Silva
- **Defenders:** Piquè, Ramos, Godín

First we need to identify the players exact name strings StatsBomb uses. This can be also achieved by a web search, however is a serious user experience concern for the future caused by the StatsBomb database. 


In [44]:
# We can get the real names of all these players from the 2015/16 data since all of them
# were active in that season (competition_id=11, season_id=27).

# We get the match data of Barcelona, Real Madrid and Atletico Madrid to extract players
barca_match_id = sb_client.get_matches(competition_id=11, season_id=27,team = "Barcelona"
                                    ).iloc[0]['match_id']

real_match_id = sb_client.get_matches(competition_id=11, season_id=27,team = "Real Madrid"
                                    ).iloc[0]['match_id']

atleti_match_id = sb_client.get_matches(competition_id=11, season_id=27,team = "Atlético Madrid"
                                    ).iloc[0]['match_id']
print(f"Barcelona match_id:      {barca_match_id}")
print(f"Real Madrid match_id:    {real_match_id}")
print(f"Atletico match_id:       {atleti_match_id}")

Barcelona match_id:      266236
Real Madrid match_id:    3825567
Atletico match_id:       3825563


In [50]:
def get_season_squad_names(
    competition_id: int,
    season_id: int,
    team: str,
) -> pd.DataFrame:
    """Get all players who appeared for a team across a full season."""
    matches = sb_client.get_matches(
        competition_id=competition_id,
        season_id=season_id,
        team=team,
    )
    
    all_players = []
    for match_id in matches["match_id"]:
        events = sb_client.get_events(match_id=match_id)
        team_events = events[events["team"] == team][["player", "position"]]
        all_players.append(team_events)
    
    return (
        pd.concat(all_players, ignore_index=True)
        .drop_duplicates(subset="player")
        .sort_values("player")
        .reset_index(drop=True)
    )

print("=== BARCELONA ===")
print(get_season_squad_names(11, 27, "Barcelona").to_string())

=== BARCELONA ===
                            player                   position
0            Adriano Correia Claro                  Left Back
1               Aleix Vidal Parreu                 Right Back
2             Andrés Iniesta Luján       Left Center Midfield
3                       Arda Turan       Left Center Midfield
4       Claudio Andrés Bravo Muñoz                 Goalkeeper
5            Daniel Alves da Silva                 Right Back
6       Douglas Pereira dos Santos                 Right Back
7            Gerard Gumbau Garriga    Left Defensive Midfield
8            Gerard Piqué Bernabéu          Right Center Back
9                     Ivan Rakitić      Right Center Midfield
10     Javier Alejandro Mascherano          Right Center Back
11                Jordi Alba Ramos                  Left Back
12                  Jérémy Mathieu           Left Center Back
13  Lionel Andrés Messi Cuccittini                 Right Wing
14        Luis Alberto Suárez Díaz             Cente

In [51]:
print("\n=== REAL MADRID ===")
print(get_season_squad_names(11, 27, "Real Madrid").to_string())


=== REAL MADRID ===
                                 player                   position
0                    Borja Mayoral Moya      Right Center Midfield
1              Carlos Henrique Casimiro    Left Defensive Midfield
2   Cristiano Ronaldo dos Santos Aveiro                  Left Wing
3                 Daniel Carvajal Ramos                 Right Back
4                  Danilo Luiz da Silva                 Right Back
5                       Denis Cheryshev                  Left Wing
6              Francisco Casilla Cortés                 Goalkeeper
7        Francisco Román Alarcón Suárez                 Right Wing
8                     Gareth Frank Bale  Center Attacking Midfield
9           James David Rodríguez Rubio  Center Attacking Midfield
10                  Jesé Rodríguez Ruiz             Center Forward
11      José Ignacio Fernández Iglesias           Left Center Back
12                        Karim Benzema             Center Forward
13                  Keylor Navas Gamboa  

In [52]:
print("\n=== ATLETICO MADRID ===")
print(get_season_squad_names(11, 27, "Atlético Madrid").to_string())


=== ATLETICO MADRID ===
                             player                   position
0                 Antoine Griezmann        Left Center Forward
1          Augusto Matías Fernández       Left Center Midfield
2        Claudio Matías Kranevitter  Center Defensive Midfield
3          Diego Roberto Godín Leal           Left Center Back
4         Fernando José Torres Sanz             Center Forward
5             Filipe Luís Kasmirski                  Left Back
6          Gabriel Fernández Arenas   Right Defensive Midfield
7      Guilherme Magdalena Siqueira                  Left Back
8          Ignacio Monsalve Vicente          Right Center Back
9   Jackson Arley Martínez Valencia       Right Center Forward
10                        Jan Oblak                 Goalkeeper
11               Jesús Gámez Duarte                  Left Back
12       Jorge Resurrección Merodio              Left Midfield
13     José María Giménez de Vargas          Right Center Back
14      Juan Francisco Torres 

Lionel Andrés Messi Cuccittini,  Luis Alberto Suárez Díaz, Cristiano Ronaldo dos Santos Aveiro



Luka Modrić, Sergio Busquets i Burgos, Andrés Iniesta Luján



Sergio Ramos García, Javier Alejandro Mascherano, Diego Roberto Godín Leal